# OPUS Benchmark (OPUS Data Selection)

Runs the **OPUS** (Optimizer-induced Projected Utility Selection) training benchmark.

**What this measures:** End-to-end step timing WITH OPUS scoring (ghost capture, sketch projection, Boltzmann selection). Provides `T_opus` for computing overhead.

**Run the Baseline notebook FIRST**, then run this one.

## Step 1: Verify GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No CUDA GPU detected. DeepSpeed requires NVIDIA GPU.")

## Step 2: Upload Codebase to Colab Runtime

If you already ran the baseline on this same runtime, the project is already there — just run the cell and it will skip.

Otherwise, zip the folder on your Mac first:
```bash
cd ~/Downloads && zip -r OpusImplementation_updated.zip OpusImplementation_updated/
```

In [ ]:
import os, sys, io, zipfile, time, shutil

PROJECT_DIR = "/content/OpusImplementation_updated"
MARKER = os.path.join(PROJECT_DIR, "requirements.txt")

# Clean up any stale/partial directory from previous failed attempts
if os.path.exists(PROJECT_DIR) and not os.path.isfile(MARKER):
    print(f"Removing stale directory: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

if os.path.isfile(MARKER):
    print(f"Project already exists at {PROJECT_DIR}, skipping upload.")
else:
    import ipywidgets as widgets
    from IPython.display import display

    print("Select OpusImplementation_updated.zip using the button below.")
    uploader = widgets.FileUpload(accept=".zip", multiple=False)
    display(uploader)

    # Wait for user to pick a file (poll every 2s, timeout 5 min)
    for i in range(150):
        time.sleep(2)
        if isinstance(uploader.value, (list, tuple)) and len(uploader.value) > 0:
            break
        elif isinstance(uploader.value, dict) and len(uploader.value) > 0:
            break
    else:
        raise TimeoutError("No file selected after 5 minutes. Re-run this cell.")

    # Extract content (handle v7 and v8 API differences)
    if isinstance(uploader.value, dict):
        fname = list(uploader.value.keys())[0]
        raw = uploader.value[fname]["content"]
    else:
        raw = bytes(uploader.value[0]["content"])
        fname = uploader.value[0]["name"]

    print(f"Received: {fname} ({len(raw) / 1e6:.1f} MB)")
    print("Extracting...")
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        z.extractall("/content/")

    if not os.path.isfile(MARKER):
        print("WARNING: requirements.txt not at expected path. Zip contents:")
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            for name in z.namelist()[:20]:
                print(f"  {name}")
        raise FileNotFoundError(
            f"Expected {MARKER}. "
            "Make sure the zip contains 'OpusImplementation_updated/' as the top-level folder."
        )
    print("Extraction complete!")

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

OUTPUT_DIR = os.path.join(PROJECT_DIR, "benchmark_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
DATA_DIR = os.path.join(PROJECT_DIR, "synth_local_en")

print(f"\nProject dir: {PROJECT_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Files:")
!ls {PROJECT_DIR}

In [ ]:
# ============================================================
# FALLBACK: If the widget above doesn't work, uncomment one:
# ============================================================

# Option A: gdown from Google Drive shared link
# !pip install -q gdown
# !gdown --fuzzy "YOUR_DRIVE_SHARE_LINK_HERE" -O /content/opus.zip
# !unzip -qo /content/opus.zip -d /content/ && rm /content/opus.zip

# Option B: wget from any direct download URL
# !wget -q "YOUR_DIRECT_URL_HERE" -O /content/opus.zip
# !unzip -qo /content/opus.zip -d /content/ && rm /content/opus.zip

## Step 3: Install Dependencies

In [ ]:
!pip install transformers==4.40.0 datasets deepspeed ninja triton huggingface_hub flash-linear-attention 2>&1 | tail -5
print("\nDependencies installed.")

## Step 3b: Verify Import Chain

Checks that all modules the trainer needs can be imported from the Colab runtime.

In [ ]:
import os, sys

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

errors = []

# 1. Core files must exist
for f in ["recurrence_model_1b.py", "recurrence_model_70b.py", "training.py",
           "data.py", "liger_ops.py", "kernels/__init__.py",
           "production/trainer.py", "production/train_prod.py",
           "production/config.py", "production/opus/__init__.py",
           "production/deepspeed_zero2_config.json"]:
    path = os.path.join(PROJECT_DIR, f)
    status = "OK" if os.path.isfile(path) else "MISSING"
    if status == "MISSING":
        errors.append(f)
    print(f"  [{status}] {f}")

# 2. Test key Python imports
print("\nImport checks:")
for mod_name in ["production.config", "production.opus", "data", "training",
                  "recurrence_model_1b", "liger_ops", "kernels"]:
    try:
        __import__(mod_name)
        print(f"  [OK] import {mod_name}")
    except Exception as e:
        errors.append(f"import {mod_name}: {e}")
        print(f"  [FAIL] import {mod_name}: {e}")

# 3. Test DeepSpeed
print("\nDeepSpeed check:")
try:
    import deepspeed
    print(f"  [OK] deepspeed {deepspeed.__version__}")
except Exception as e:
    errors.append(f"deepspeed: {e}")
    print(f"  [FAIL] {e}")

if errors:
    print(f"\n{len(errors)} issue(s) found — fix before running benchmark.")
else:
    print("\nAll checks passed. Ready to run.")

## Step 4: Download Training Data (PleIAs/SYNTH shard)

In [ ]:
import os

if not os.path.exists(os.path.join(DATA_DIR, "dataset_info.json")):
    !python {PROJECT_DIR}/scripts/download_synth_shard.py -o {DATA_DIR}
    print("Synth data downloaded.")
else:
    print(f"Synth data already exists at {DATA_DIR}, skipping.")

## Step 5: Create Matched OPUS Config

Identical to the Baseline notebook **except** `selection_mode = "opus"`.

Both notebooks use: 1B model, vocab 131072, seq_len 512, candidate_multiplier 32, 100 steps.

In [ ]:
import json, os

config = {
    "model": {
        "target": "1b",
        "embedding_type": "standard",
        "vocab_size": 131072,
        "hidden_size": None,
        "num_layers": None,
        "num_real_experts": None,
        "num_null_experts": None,
        "top_k": None,
        "data_sparsity": None
    },
    "data": {
        "dataset_name": "PleIAs/SYNTH",
        "local_path": DATA_DIR,
        "filter_language": "en",
        "seed": 42,
        "num_workers": 2
    },
    "train": {
        "total_steps": 100,
        "learning_rate": 0.0002,
        "weight_decay": 0.0,
        "betas": [0.9, 0.95],
        "eps": 1e-08,
        "grad_clip": 1.0,
        "seq_len_train": 512,
        "micro_batch_size": 1,
        "grad_accum_steps": 1,
        "checkpoint_every": 250,
        "log_every": 10,
        "checkpoint_dir": os.path.join(OUTPUT_DIR, "checkpoints_opus"),
        "step_metrics_log_path": os.path.join(OUTPUT_DIR, "step_timing_opus.jsonl"),
        "step_metrics_log_every": 1,
        "deepspeed_config": "production/deepspeed_zero2_config.json",
        "use_bf16": True
    },
    "opus": {
        "enabled": True,
        "selection_mode": "opus",
        "candidate_multiplier": 32,
        "selection_ratio": 0.5,
        "score_seq_len": 512,
        "proxy_batch_size": 8,
        "sketch_dim": 8192,
        "temperature": 0.9,
        "sketch_seed": 42,
        "fallback_random_on_error": True,
        "max_selector_time_s": 30.0,
        "include_embeddings": False,
        "include_lm_head": False,
        "score_dtype": "bf16",
        "track_nonfinite_stats": False,
        "zero2_exact_global_scoring": True,
        "strict_shard_preconditioner": True,
        "benchmark_dual_mode": False,
        "benchmark_every": 100,
        "benchmark_warmup_steps": 20,
        "benchmark_log_path": os.path.join(OUTPUT_DIR, "selector_benchmark_opus.jsonl")
    }
}

config_path = os.path.join(PROJECT_DIR, "config_colab_opus.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config written to: {config_path}")
print(f"Selection mode:    {config['opus']['selection_mode']}")
print(f"Total steps:       {config['train']['total_steps']}")
print(f"Seq len:           {config['train']['seq_len_train']}")
print(f"Vocab size:        {config['model']['vocab_size']}")
print(f"Candidate mult:    {config['opus']['candidate_multiplier']}")
print(f"Sketch dim:        {config['opus']['sketch_dim']}")

## Step 6: Run OPUS Benchmark

Runs production trainer with **OPUS selection** (ghost capture + sketch projection + Boltzmann selection).

Each step is slower than baseline due to the scoring pass — that overhead is what we measure.

100 steps at ~8-20s each = ~13-33 minutes.

In [ ]:
import subprocess, os

log_file = os.path.join(OUTPUT_DIR, "opus_run.log")

cmd = (
    f"cd '{PROJECT_DIR}' && "
    f"PYTHONPATH='{PROJECT_DIR}' "
    f"deepspeed --num_gpus 1 "
    f"production/train_prod.py "
    f"--config config_colab_opus.json"
)

print(f"Running: {cmd}")
print(f"Logging to: {log_file}")
print("=" * 60)

process = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

with open(log_file, "w") as f:
    for line in process.stdout:
        print(line, end="")
        f.write(line)

process.wait()
print(f"\nProcess exited with code: {process.returncode}")

## Step 7: View OPUS Timing Results

In [ ]:
import json, statistics, os

log_path = os.path.join(OUTPUT_DIR, "step_timing_opus.jsonl")
warmup = 20

steps = []
with open(log_path, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            steps.append(json.loads(line))

print(f"Total steps logged: {len(steps)}")
print(f"\n--- First 5 steps ---")
for s in steps[:5]:
    print(json.dumps(s, indent=2))

post_warmup = [s for s in steps if s["step"] >= warmup]
if post_warmup:
    times = [s["step_total_s"] for s in post_warmup]
    print(f"\n--- OPUS Timing (post warmup, {len(times)} steps) ---")
    print(f"  Mean:   {statistics.fmean(times):.3f} s/step")
    print(f"  Median: {statistics.median(times):.3f} s/step")
    sorted_times = sorted(times)
    p90_idx = min(len(sorted_times)-1, int(0.9 * (len(sorted_times)-1)))
    print(f"  P90:    {sorted_times[p90_idx]:.3f} s/step")
    print(f"  Min:    {min(times):.3f} s/step")
    print(f"  Max:    {max(times):.3f} s/step")

    scoring_times = [s.get("scoring_pass_s", 0) for s in post_warmup]
    feature_times = [s.get("feature_build_s", 0) for s in post_warmup]
    selector_times = [s.get("selector_total_s", 0) for s in post_warmup]
    train_times = [s.get("train_pass_s", 0) for s in post_warmup]

    if any(t > 0 for t in scoring_times):
        print(f"\n--- OPUS Component Breakdown (median) ---")
        print(f"  Scoring pass:  {statistics.median(scoring_times):.3f} s")
        print(f"  Feature build: {statistics.median(feature_times):.3f} s")
        print(f"  Selector:      {statistics.median(selector_times):.3f} s")
        print(f"  Train pass:    {statistics.median(train_times):.3f} s")
else:
    print("No steps after warmup!")

## Step 8: Compute Overhead Report (OPUS vs Baseline)

Both timing logs should be in `benchmark_outputs/` (from running both notebooks on the same runtime).

In [ ]:
import os

baseline_log = os.path.join(OUTPUT_DIR, "step_timing_baseline.jsonl")
opus_log = os.path.join(OUTPUT_DIR, "step_timing_opus.jsonl")
report_path = os.path.join(OUTPUT_DIR, "overhead_report.json")

if not os.path.exists(baseline_log):
    print(f"ERROR: Baseline log not found at: {baseline_log}")
    print("Run the baseline notebook first on this same runtime!")
else:
    !python {PROJECT_DIR}/scripts/report_opus_overhead.py \
        --baseline-log "{baseline_log}" \
        --opus-log "{opus_log}" \
        --warmup-steps 20 \
        --output-json "{report_path}"

In [ ]:
import json, os

report_path = os.path.join(OUTPUT_DIR, "overhead_report.json")

with open(report_path) as f:
    report = json.load(f)

print("=" * 60)
print("       OPUS OVERHEAD REPORT (Paper-Style)")
print("=" * 60)
print(f"\nSteps compared:  {report['steps_compared']}")
print(f"Warmup skipped:  {report['warmup_steps']}")
print(f"\nBaseline (random selection):")
print(f"  Median step time: {report['baseline']['step_total_s']['median']:.3f} s")
print(f"  Mean step time:   {report['baseline']['step_total_s']['mean']:.3f} s")
print(f"\nOPUS (scored selection):")
print(f"  Median step time: {report['opus']['step_total_s']['median']:.3f} s")
print(f"  Mean step time:   {report['opus']['step_total_s']['mean']:.3f} s")
print(f"\n{'=' * 60}")
print(f"  OVERHEAD: {report['paper_style_overhead_percent']:.2f}%")
print(f"{'=' * 60}")
print(f"\n(Paper reports ~4.7% overhead for 70B model)")
print(f"(1B model overhead will differ due to different compute/scoring ratio)")
print(f"\nFull report saved at: {report_path}")